# Communicating with servers via HTTP

## `requests`

`requests` is a third-party package that faciliates the task of sending HTTP requests to communicate with servers.

Whilst `requests` can be used to retrieve HTML from web pages, it shines when it comes to communicating with web APIs that require several arguments.

Let's first import `requests` and retrieve a web page.

We can call `get()` with an URL and save the `Response` object.

In [1]:
import requests

# Get the response from a page
response = requests.get("https://cambridgespark.com")

Let's check the status code, `200` means `OK` -- the request was successful.

In [2]:
response.status_code

200

We can also access the content of the page with the attribute `.content`

In [3]:
response.content

b'<!doctype html><html lang="en"><head>\n    <meta charset="utf-8">\n    <title>AI &amp; Data Apprenticeships | Cambridge Spark</title>\n    <link rel="shortcut icon" href="https://www.cambridgespark.com/hubfs/cambridge-spark-2023-favicon_48x48px.webp">\n    <meta name="description" content="Unlock business value with Cambridge Spark. We deliver data science and AI apprenticeships designed to build a highly skilled, future-ready workforce.">\n\n    <meta property="og:title" content="AI &amp; Data Apprenticeships | Cambridge Spark"> \n    <meta property="og:description" content="Unlock business value with Cambridge Spark. We deliver data science and AI apprenticeships designed to build a highly skilled, future-ready workforce."> \n    <meta property="og:url" content="https://www.cambridgespark.com"> \n\n    <meta name="twitter:title" content="AI &amp; Data Apprenticeships | Cambridge Spark"> \n    <meta name="twitter:description" content="Unlock business value with Cambridge Spark. We d

Let's look at how `requests` helps when we work with more complex web APIs.

We've created our own Banking Web API.

We define below the base URL for the API so you do not have to type the whole URL everytime and can just append the endpoint you want to call.

In [4]:
BASE_URL = "http://127.0.0.1:8080/banking"

Find below the Documentation for the API:

## Bank of Cambridge Spark -- API Documentation

Each endpoint can be called by appending to the base url: `http://localhost:8080/banking`

### Authentication

To authenticate you will need an API token.

Once you have it, pass: {"Authorization": "your_token"} to the headers of any request.

#### List all users

| Endpoint     | Method  | Returns                | Description
| :-----------:| :-----: | :--------------------: | :----------------------: |
| /api/users   | GET     | {user_id: name, ...}   |  Get a list of all users |


Example: `{{ BASE_URL }}/api/users`

#### List all transactions for user


| Endpoint     | Method  | Returns                | Description | Optional Parameters
| :-----------:| :-----: | :--------------------: | :----------------------: | :---: |
| /api/get_transactions/(user_id)    | GET     | [{'amount': 100.0, 'type': 'CREDIT'}, ...]  |  Get a list of all transaction for user | type=CREDIT or type=DEBIT to filter |

Example: `{{ BASE_URL }}/api/get_transactions/1?type=CREDIT`

#### Add user

| Endpoint     | Method  | Returns                | Description
| :-----------:| :-----: | :--------------------: | :----------------------: |
| /api/add_user   | POST     | {"name": "username"}   | Create a new user |

Example: `{{ BASE_URL }}/api/add_user` with {"name": "John"} as parameter.


#### Add transactions for user

| Endpoint     | Method  | Returns                | Description
| :-----------:| :-----: | :--------------------: | :----------------------: |
| /api/add_transaction/(user_id)   | POST     | {"amount": value, "type": CREDIT or DEBIT}   | Add a transaction for a user |
 	
Example: `{{ BASE_URL }}/api/add_transaction/1` with {"amount": 10, "type": "CREDIT"} as parameter.

Note: here the amount needs to be a float or integer, and type needs to be either CREDIT or DEBIT. A code 400 will be returned otherwise.

## Let's get started!

To start with, we will try to list all available users.

To do so, we can call the `/api/users` endpoint.

Let's call our first endpoint to get all available users:

In [5]:
url_endpoint = f"{BASE_URL}/api/users"

r = requests.get(url=url_endpoint)

In [6]:
# Let's verify we got a status code 200 first
r.status_code

401

Oops, error [401](https://httpstatuses.com/401) stands for `Unauthorized` - HTTP is a well defined protocol where each error corresponds to a specific issue. 

We can check the content returned by the API for more details:

In [7]:
r.content

b'Invalid token.'

We "forgot" to mention that our API requires authentication... (see documentation for more details on how to authenticate)

Users need to pass an API key in the headers to authenticate, you can see it as a password. We provide the API key below.

Note: with actual APIs, the documentation will explain how to pass a key to authenticate, it often follows a similar process we are using here.

Thanksfully `requests` allows us to easily define the headers we want to pass when making a request:

In [8]:
api_key = {"Authorization": "NRCqpfD3"}

Let's try again, with our API key this time

In [9]:
url_endpoint = f"{BASE_URL}/api/users"

r = requests.get(url=url_endpoint, headers=api_key)

r.status_code

200

Status code is [200](https://httpstatuses.com/200), great!

Let's see the content:

In [10]:
r.content

b'{"1":"Caroline","2":"Marium"}\n'

Since our API return JSON objects, we can now use `.json()` directly from `requests` to load the data as a dictionary.

In [11]:
users = r.json()
users

{'1': 'Caroline', '2': 'Marium'}

### Exercise

First we will get transactions for a specific user - as you can see in the documentation, you can use the `get_transactions` endpoint here and eppend the user_id of the user you want to retrieve data from

Get transaction data for our first user (check the endpoint in the documentation, and don't forget the API key)

As you can see in the documentation, you can add an optional parameter to your request. With APIs you usually add a `?` followed by the optional arguments you want to add:

In [12]:
user_id = 1
url_endpoint = f"{BASE_URL}/api/get_transactions/{user_id}?type=CREDIT"

r = requests.get(url=url_endpoint, headers=api_key)
r.json()

[{'amount': 1000.0, 'type': 'CREDIT'},
 {'amount': 1000.0, 'type': 'CREDIT'},
 {'amount': 1000.0, 'type': 'CREDIT'},
 {'amount': 1123581321345.55, 'type': 'CREDIT'},
 {'amount': 765.0, 'type': 'CREDIT'},
 {'amount': 1.0, 'type': 'CREDIT'},
 {'amount': 1.0, 'type': 'CREDIT'},
 {'amount': 1.0, 'type': 'CREDIT'},
 {'amount': 1.0, 'type': 'CREDIT'},
 {'amount': 11000.0, 'type': 'CREDIT'}]

This can quickly become harder to work with as you are adding more optional parameters. `requests` provides a better way to add such parameters.

### Exercise

Instead of using `?` followed by the parameters, create a new dictionary with the arguments you want to use and their value, then pass this dictionary to `get()` as keyword argument `params`.

Syntax:

```
requests.get(your_url, params=your_parameters, headers=your_headers)
```

Our web API also supports some `POST` operations where you can add data to our database. The first one we will see here is `add_user` that allows you to add a new user.

With `requests` we can simply use the `.post()` method and pass the data we want to send as a `data` parameter.

Replace the name in the dictionary below by your own name. This defines the data about a user that we want to send to the API.

In [13]:
# Replace by a name of your choice
user = {"name": "John"}

In [14]:
# Here we have a new argument, data, that allows us to post data to the API
url_endpoint = f"{BASE_URL}/api/add_user"

r = requests.post(url_endpoint, json=user, headers=api_key)

Let's check the status code:

In [15]:
r.status_code

200

`200` means the operation was successful! If we call the get_users endpoint again (like we did at the very beginning) we should see our new user.

Get the updated list of users to find your new user's ID:

Finally, our web API allows us to add transactions by sending data serialised in json through a `POST` request to the same endpoint we've seen before. With `requests` we can simply use the `.post()` method and pass the data we want to send as a `data` parameter.

In [16]:
# Same endpoint as before, but here we'll use a POST request
user_id = 2 
url_endpoint = f"{BASE_URL}/api/add_transaction/{user_id}"

# The advantage of using our own API is that we can credit accounts as we want :)
transaction_to_add = {"type": "CREDIT", "amount": 1000}

r = requests.post(url_endpoint, json=transaction_to_add, headers=api_key)

In [17]:
# We verify the status code
r.status_code

200

### Exercise

Let's check that our transaction was added; call the endpoint to `get_transactions` for the account you credited.

- use a `.get()` request with the necessary `headers` and the same `user_id` as used above

### [Optional] A more advanced exercise

By now you should have all the tools you need to write your own programmes that leverage the power of APIs and automate tasks.

For instance here you should be able to write a simple programme that compute the balance for a given user.

For this you will need to:
- retrieve all CREDIT transactions for a user
- compute the sum of credits
- retrieve all DEBIT transactions for a user
- compute the sum of debit
- compute the balance

In [18]:
# Add your code below to compute the balance for a given user!

